# 08. 멀티 에이전트 패턴

## 학습 목표

5가지 멀티 에이전트 패턴을 이해하고 구현합니다.

이 노트북에서 다루는 내용:
- **Subagents**: 메인 에이전트가 전문 서브에이전트를 도구로 호출
- **Handoffs**: `Command(goto=...)`로 에이전트 간 상태 전환
- **Skills**: 단일 에이전트가 상황에 따라 전문 프롬프트를 로드
- **Router**: 분류기가 입력을 적절한 에이전트로 라우팅
- **Custom**: 개발자가 완전히 제어하는 복잡한 워크플로

## 8.1 환경 설정

In [1]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-4.1",
)

print("모델 준비 완료:", model.model_name)

모델 준비 완료: gpt-4.1


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


## 8.2 멀티 에이전트 패턴 비교

아래 표는 5가지 멀티 에이전트 패턴을 비교합니다. 각 패턴은 서로 다른 상황에 적합하며, 프로젝트의 요구사항에 맞게 선택해야 합니다.

| 패턴 | 라우팅 주체 | 상태 공유 | 적합한 상황 |
|------|-----------|----------|------------|
| **Subagents** | 메인 에이전트 | 도구로 격리 | 병렬 처리, 분산 개발 |
| **Handoffs** | 도구 호출 | 상태 전환 | 순차적 멀티홉 |
| **Skills** | 단일 에이전트 | 프롬프트 교체 | 도메인 특화 |
| **Router** | 분류기 | 병렬 실행 | 멀티 도메인 |
| **Custom** | 개발자 정의 | 완전 제어 | 복잡한 워크플로 |

### 핵심 차이점
- **Subagents**는 각 서브에이전트가 독립적으로 실행되어 결과만 반환합니다.
- **Handoffs**는 대화 상태가 에이전트 간에 전달됩니다.
- **Skills**는 하나의 에이전트가 여러 역할을 전환합니다.
- **Router**는 입력을 분류한 뒤 적절한 에이전트에게 위임합니다.

## 8.3 서브에이전트 패턴

메인 에이전트(감독자)가 전문 서브에이전트를 **도구로** 호출하는 패턴입니다.

### 특징
- 각 서브에이전트는 도구 함수로 캡슐화됩니다.
- 메인 에이전트가 어떤 서브에이전트를 호출할지 판단합니다.
- 서브에이전트의 내부 상태는 메인 에이전트와 격리됩니다.
- 병렬 실행이 가능하여 성능에 유리합니다.

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool

# 전문 도구 정의
@tool
def math_expert(question: str) -> str:
    """수학 문제를 풀어줍니다. 수학적 계산이 필요할 때 사용하세요."""
    # 실제로는 계산 로직이 들어갑니다
    return f"수학 답변: '{question}'에 대한 답이 계산되었습니다."

@tool
def code_expert(question: str) -> str:
    """프로그래밍 질문에 답합니다. 코딩 관련 질문에 사용하세요."""
    return f"코드 답변: '{question}'에 대한 솔루션입니다"

@tool
def general_search(query: str) -> str:
    """일반 정보를 검색합니다."""
    return f"'{query}'에 대한 검색 결과"

# 감독자(supervisor) 에이전트
supervisor = create_agent(
    model=model,
    tools=[math_expert, code_expert, general_search],
    system_prompt="""당신은 전문가에게 작업을 위임하는 감독 에이전트입니다:
- 수학 문제는 math_expert를 사용하세요
- 프로그래밍 질문은 code_expert를 사용하세요
- 그 외에는 general_search를 사용하세요
항상 가장 적절한 전문가에게 위임하세요.""",
)

result = supervisor.invoke(
    {"messages": [{"role": "user", "content": "10의 팩토리얼은 얼마인가요?"}]},
    config=lf_config,
)
print("서브에이전트 결과:", result["messages"][-1].content)

서브에이전트 결과: 10의 팩토리얼 값은 3,628,800입니다.


## 8.4 핸드오프 패턴

`Command(goto=...)`로 에이전트 간 **상태를 전환**하는 패턴입니다.

### 특징
- 도구가 `Command` 객체를 반환하여 다른 에이전트로 전환합니다.
- 대화 상태(메시지 히스토리)가 다음 에이전트로 전달됩니다.
- `StateGraph`를 사용하여 에이전트 간 흐름을 정의합니다.
- 고객 서비스의 부서 이관 같은 순차적 멀티홉 시나리오에 적합합니다.

In [4]:
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END, MessagesState

# 핸드오프를 위한 도구
@tool
def transfer_to_sales() -> Command:
    """대화를 영업 부서로 전환합니다."""
    return Command(goto="sales_agent", graph=Command.PARENT)

@tool
def transfer_to_support() -> Command:
    """대화를 기술 지원 부서로 전환합니다."""
    return Command(goto="support_agent", graph=Command.PARENT)

@tool
def resolve_query(answer: str) -> str:
    """사용자에게 최종 답변을 제공합니다."""
    return answer

# 라우터 에이전트
router_agent = create_agent(
    model=model,
    tools=[transfer_to_sales, transfer_to_support],
    system_prompt="당신은 안내 데스크입니다. 고객을 적절한 부서로 안내하세요.",
    name="router",
)

# 영업 에이전트
sales_agent = create_agent(
    model=model,
    tools=[resolve_query],
    system_prompt="당신은 영업 에이전트입니다. 가격 및 제품 문의를 도와주세요.",
    name="sales_agent",
)

# 지원 에이전트
support_agent = create_agent(
    model=model,
    tools=[resolve_query],
    system_prompt="당신은 기술 지원 에이전트입니다. 기술적 문제를 도와주세요.",
    name="support_agent",
)

# 핸드오프 그래프 구성
builder = StateGraph(MessagesState)
builder.add_node(router_agent)
builder.add_node(sales_agent)
builder.add_node(support_agent)
builder.add_edge(START, "router")

graph = builder.compile()

result = graph.invoke(
    {"messages": [{"role": "user", "content": "엔터프라이즈 플랜 가격을 알고 싶습니다."}]},
    config=lf_config,
)
print("핸드오프 결과:", result["messages"][-1].content)

핸드오프 결과: 엔터프라이즈 플랜 가격 문의 주셔서 감사합니다.

엔터프라이즈 플랜은 고객님의 요구 사항에 따라 맞춤 견적이 제공됩니다. 사용하실 인원 수, 필요한 기능, 또는 요청 사항을 알려주시면 보다 정확한 견적을 안내해 드릴 수 있습니다.

추가 정보를 공유해 주시면 맞춤 견적서를 전달드리겠습니다.


## 8.5 스킬 패턴

단일 에이전트가 상황에 따라 **전문 프롬프트를 로드**하는 패턴입니다.

### 특징
- 하나의 에이전트가 여러 "스킬"을 가집니다.
- 각 스킬은 특화된 시스템 프롬프트입니다.
- 에이전트가 필요한 스킬을 동적으로 로드합니다.
- 여러 에이전트를 관리할 필요 없이 하나의 에이전트로 다양한 작업을 처리할 수 있습니다.

In [5]:
# 스킬 정의
skills = {
    "translator": "당신은 전문 번역가입니다. 언어 간 텍스트를 정확하게 번역하세요.",
    "summarizer": "당신은 전문 요약가입니다. 긴 텍스트를 간결하게 요약하세요.",
    "coder": "당신은 전문 프로그래머입니다. 깔끔하고 효율적인 코드를 작성하세요.",
}

@tool
def load_skill(skill_name: str) -> str:
    """전문 스킬을 로드합니다. 사용 가능한 스킬: translator, summarizer, coder."""
    if skill_name in skills:
        return f"스킬 로드 완료: {skill_name}. 지침: {skills[skill_name]}"
    return f"알 수 없는 스킬: {skill_name}. 사용 가능: {list(skills.keys())}"

skill_agent = create_agent(
    model=model,
    tools=[load_skill],
    system_prompt="""당신은 전문 기술에 접근할 수 있는 다재다능한 어시스턴트입니다.
사용자의 요청을 처리하기 전에 적절한 스킬을 로드하세요.""",
)

result = skill_agent.invoke(
    {"messages": [{"role": "user", "content": "'Hello World'를 한국어와 일본어로 번역해 주세요."}]},
    config=lf_config,
)
print("스킬 패턴 결과:", result["messages"][-1].content)

스킬 패턴 결과: "Hello World"를 한국어와 일본어로 번역해 드리겠습니다.

- 한국어: 안녕하세요, 세계!
- 일본어: こんにちは、世界！

추가로 번역이 필요한 문장이 있다면 말씀해 주세요.


## 8.6 라우터 패턴

분류기가 입력을 적절한 에이전트로 **라우팅**하는 패턴입니다.

### 특징
- 먼저 쿼리를 분류(classify)합니다.
- 분류 결과에 따라 적절한 전문 에이전트(도구)로 위임합니다.
- 멀티 도메인 시스템에서 유용합니다.
- 분류 로직은 규칙 기반 또는 LLM 기반으로 구현할 수 있습니다.

In [6]:
from langgraph.types import Send

@tool
def classify_query(query: str) -> str:
    """쿼리를 카테고리로 분류합니다: math, code, general."""
    query_lower = query.lower()
    if any(w in query_lower for w in ["calculate", "math", "sum", "multiply"]):
        return "math"
    elif any(w in query_lower for w in ["code", "program", "function", "python"]):
        return "code"
    return "general"

# 라우터: 분류 결과에 따라 전문 에이전트로 전달
router = create_agent(
    model=model,
    tools=[classify_query, math_expert, code_expert, general_search],
    system_prompt="""당신은 라우팅 에이전트입니다. 먼저 쿼리를 분류한 다음 적절한 전문가에게 전달하세요:
- 수학 쿼리 -> math_expert 사용
- 코드 쿼리 -> code_expert 사용
- 일반 쿼리 -> general_search 사용""",
)

result = router.invoke(
    {"messages": [{"role": "user", "content": "리스트를 정렬하는 Python 함수를 작성해 주세요."}]},
    config=lf_config,
)
print("라우터 결과:", result["messages"][-1].content)

라우터 결과: 리스트를 정렬하는 Python 함수에 대해 문의해주셨네요. 아래는 예시 코드입니다:

```python
def sort_list(lst):
    return sorted(lst)
```

이 함수에 리스트를 넘기면, 정렬된 새로운 리스트를 반환합니다.


## 8.7 패턴 선택 가이드

어떤 멀티 에이전트 패턴을 선택해야 할까요? 아래 가이드를 참고하세요.

### 결정 트리

1. **에이전트가 독립적으로 작업 가능한가?**
   - YES -> **Subagents** (병렬 실행, 결과 취합)
   - NO -> 다음 질문으로

2. **대화 상태가 에이전트 간에 전달되어야 하는가?**
   - YES -> **Handoffs** (상태 전환, 멀티홉)
   - NO -> 다음 질문으로

3. **하나의 에이전트가 여러 역할을 전환하면 되는가?**
   - YES -> **Skills** (프롬프트 교체)
   - NO -> 다음 질문으로

4. **입력을 분류한 뒤 적절한 처리기로 보내면 되는가?**
   - YES -> **Router** (분류 후 위임)
   - NO -> **Custom** (완전 커스텀 그래프)

### 실용적 권장사항
- 처음에는 **가장 단순한 패턴**(Subagents 또는 Skills)으로 시작하세요.
- 요구사항이 복잡해지면 Handoffs나 Router로 전환하세요.
- Custom 패턴은 다른 패턴으로 해결할 수 없을 때만 사용하세요.
- 패턴을 **조합**할 수도 있습니다 (예: Router + Handoffs).

## 8.8 요약

이 노트북에서 학습한 핵심 내용:

| 패턴 | 핵심 API | 사용 시기 |
|------|---------|----------|
| **Subagents** | `create_agent` + 도구 함수 | 독립적 병렬 작업 |
| **Handoffs** | `Command(goto=...)`, `StateGraph` | 상태 전환이 필요한 멀티홉 |
| **Skills** | 도구로 프롬프트 로드 | 단일 에이전트 다중 역할 |
| **Router** | 분류 도구 + 전문 도구 | 멀티 도메인 분류 |
| **Custom** | `StateGraph` 완전 제어 | 복잡한 비즈니스 로직 |

### 핵심 원칙
- 단순한 것부터 시작하고, 필요에 따라 복잡도를 높이세요.
- 각 에이전트는 **하나의 책임**만 가지도록 설계하세요.
- 에이전트 간 **인터페이스**(입력/출력)를 명확히 정의하세요.

### 다음 단계
→ **[09_custom_workflow_and_rag.ipynb](./09_custom_workflow_and_rag.ipynb)**: 커스텀 워크플로와 RAG를 배웁니다.
